In [5]:
import geopandas as gpd
import pandas as pd

one_drive_path = "C:/Users/JKolberg/OneDrive - PSRC/GIS - Projects/FLU"

#### Comparison of zones in FLU GIS layer and zones in FLU table

In [7]:
# get gis layer
# flu_gdb_path = f"{one_drive_path}/FLU_draft2.gdb"
# flu_layer = "FLU2025"
# flu_shp = gpd.read_file(flu_gdb_path, layer=flu_layer)
flu_shp_path = "Q:/Projects/2023_Baseyear/FLU_and_Lockouts/GIS/FLU_2025/FLU_20260512/QC/flu2025_dis.shp"
flu_shp = gpd.read_file(flu_shp_path)

# get table
flu_table_path = f"{one_drive_path}/Zoning_2026_d3.xlsx"
flu_table = pd.read_excel(flu_table_path)

In [8]:
flu_shp.columns

Index(['Juris', 'zone_psrc', 'Juris_zn', 'geometry'], dtype='str')

In [9]:
flu_shp.loc[flu_shp['Juris'] == 'Auburn']['Juris_zn'].unique()

<StringArray>
[                     'Auburn_C-1',                      'Auburn_C-2',
                     'Auburn_C-AG',                      'Auburn_DUC',
                  'Auburn_DUC_125',                   'Auburn_DUC_75',
                  'Auburn_DUC_C-1',                  'Auburn_DUC_C-2',
                   'Auburn_DUC_FR',                   'Auburn_DUC_HW',
                   'Auburn_DUC_M1',                   'Auburn_DUC_NR',
                        'Auburn_I',                      'Auburn_L-F',
 'Auburn_LAKELAND_HILLS_SOUTH_PUD',                      'Auburn_M-2',
                      'Auburn_M-1',                       'Auburn_OS',
                      'Auburn_P-1',                      'Auburn_PUD',
                      'Auburn_R-1',                      'Auburn_R-2',
                      'Auburn_R-3',                      'Auburn_R-4',
                      'Auburn_R-F',                    'Auburn_R-MHC',
                     'Auburn_R-NM',                       'Aubu

In [4]:
# compare juris_zn values in shp and table
shp_juris = pd.Series(flu_shp['Juris_zn'].dropna().unique(), name='juris_zn')
table_juris = pd.Series(flu_table['juris_zn'].dropna().unique(), name='juris_zn')

comparison = (
    pd.DataFrame({'juris_zn': sorted(set(shp_juris) | set(table_juris))})
    .assign(
        in_shp=lambda df: df['juris_zn'].isin(shp_juris),
        in_table=lambda df: df['juris_zn'].isin(table_juris)
    )
)

comparison['status'] = comparison.apply(
    lambda row: 'both' if row['in_shp'] and row['in_table']
    else 'only_shp' if row['in_shp']
    else 'only_table',
    axis=1
)

today = pd.Timestamp.today().strftime('%Y-%m-%d')
comparison.to_excel(f"{one_drive_path}/juris_zn_shp_to_table_comparison_{today}.xlsx", index=False)

#### Fuzzy matches zone names in FLU GIS layer to FLU table

In [ ]:
# fuzzy matching for zones in gis layer but not in table

import re
from difflib import SequenceMatcher
from pathlib import Path

FUZZY_AUTO_CUTOFF = 1.0   # auto-accept fuzzy matches at/above this ratio
FUZZY_MIN_CUTOFF = 0.4     # report candidates above this; below => no suggestion
JURIS_MATCH_CSV = (Path(__file__).parent if "__file__" in globals() else Path.cwd()) \
    / "juris_zn_shp_to_table_fuzzy.csv"


def _normalize(s) -> str:
    s = str(s).strip().lower()
    return re.sub(r"[-_\s,/]+", "", s)


def fuzzy_match_juris(shp_values, table_values, manual_path: Path):
    """For each shp value with no exact table match, find best fuzzy candidate.

    Reads manual overrides from `manual_path` if it exists (columns:
    juris_zn_shp, manual_match, confirmed_missing) and uses them.
    Returns a DataFrame with match results.
    """
    shp_set = set(shp_values)
    table_set = set(table_values)
    missing = sorted(shp_set - table_set)

    # Load prior manual edits if present and schema matches
    prior_lookup = {}
    if manual_path.exists():
        prior = pd.read_csv(manual_path)
        if "juris_zn_shp" in prior.columns:
            ts = pd.Timestamp.now().strftime("%Y%m%d%H%M%S")
            prior.to_csv(manual_path.with_name(f"{manual_path.stem}_backup_{ts}.csv"), index=False)
            prior_lookup = prior.set_index("juris_zn_shp").to_dict("index")
        else:
            print(f"Note: {manual_path.name} exists but has unexpected schema; ignoring.")

    table_list = list(table_set)
    table_norm = {t: _normalize(t) for t in table_list}

    rows = []
    for shp_val in missing:
        norm_shp = _normalize(shp_val)

        # Best fuzzy candidate in table_juris
        best_score = 0.0
        best_cand = None
        for t in table_list:
            score = SequenceMatcher(None, norm_shp, table_norm[t]).ratio()
            if score > best_score:
                best_score = score
                best_cand = t

        suggested = best_cand if best_score >= FUZZY_MIN_CUTOFF else None
        auto = best_score >= FUZZY_AUTO_CUTOFF

        prior_row = prior_lookup.get(shp_val, {})
        manual_match = prior_row.get("manual_match", "")
        confirmed_missing = prior_row.get("confirmed_missing", False)
        if pd.isna(manual_match):
            manual_match = ""
        if pd.isna(confirmed_missing):
            confirmed_missing = False

        manual_str = str(manual_match).strip()
        if manual_str:
            final_match = manual_str
            match_type = "manual"
            confidence = 1.0
        elif auto:
            final_match = suggested
            match_type = "fuzzy_auto"
            confidence = round(best_score, 3)
        elif suggested is not None:
            final_match = None
            match_type = "fuzzy_review"
            confidence = round(best_score, 3)
        else:
            final_match = None
            match_type = "no_match"
            confidence = round(best_score, 3)

        if isinstance(confirmed_missing, str):
            confirmed_flag = confirmed_missing.strip().upper() in ("TRUE", "1", "YES")
        else:
            confirmed_flag = bool(confirmed_missing)

        rows.append({
            "juris_zn_shp": shp_val,
            "suggested_match": suggested,
            "confidence": confidence,
            "match_type": match_type,
            "final_match": final_match,
            "manual_match": manual_str,
            "confirmed_missing": confirmed_flag,
        })

    return pd.DataFrame(rows)


juris_matches = fuzzy_match_juris(shp_juris, table_juris, JURIS_MATCH_CSV)
juris_matches.to_csv(JURIS_MATCH_CSV, index=False)
print(f"Wrote {len(juris_matches)} rows to {JURIS_MATCH_CSV}")
print(juris_matches["match_type"].value_counts())
juris_matches
